# Milestone 1 — Formula 1 Podium Prediction Companion (CSEN903)
## Part 1 — Data Lead (P1): audit, cleaning, grain, EDA before/after, `load_clean()`

**Target:** `podium = positionOrder <= 3`  |  **Grain:** one row per `(raceId, driverId)`  |  **Prediction cutoff:** after qualifying / grid, before lights-out.

**Notebook map**
1. Repository skeleton & environment (`DATA_DIR`, seeds)
2. Load 14 tables + primary/foreign-key audit
3. EDA **before** cleaning
4. Sentinels, semantic typing, the meaning of `grid = 0`
5. Grain enforcement: duplicate `(raceId, driverId)` rows + keep-rule
6. Indianapolis 500 and DNQ / non-starters
7. EDA **after** cleaning + before/after comparison
8. Hand-off: `load_clean()` + data dictionary
9. Integration & Kaggle "Run All" verification

Every decision has a **"Why this choice?"** block directly beside the code. Numbers quoted in prose are either (a) things that follow from F1 rules or (b) explicitly marked *"confirm in the output above"* — the notebook prints the evidence so we can defend it.

---
# 1. Repository skeleton & environment setup

### Folder structure (run `setup_repo.sh`, shipped next to this notebook)
```
f1-podium/
├── data/raw/            # the 14 Kaggle CSVs (git-ignored, read-only by convention)
├── notebooks/           # one draft notebook per person: p1_data.ipynb ... p4_models.ipynb
├── main.ipynb           # merged, run-all notebook that gets submitted to Kaggle
├── src/f1_clean.py      # load_clean() and helpers (same code as section 8)
├── tools/               # merge_notebooks.py, run_all_check.sh
├── reports/             # data dictionary, figures, report PDF
├── outputs/             # local checkpoints (git-ignored)
├── requirements.txt
├── .gitignore  .gitattributes
└── README.md
```

### `.gitignore` (raw data + local checkpoints)
```
data/raw/
data/**/*.csv
outputs/
*.parquet
*.pkl
*.h5
*.pt
.ipynb_checkpoints/
__pycache__/
.venv/
.DS_Store
```

### `nbstripout`
```bash
pip install nbstripout
nbstripout --install --attributes .gitattributes   # run once PER CLONE: the filter lives in .git/config
```

### Why this choice?
* **Raw CSVs are git-ignored.** The files are large (`lap_times` alone has >500k rows), re-downloadable, and the licence is Kaggle's, not ours. Committing them bloats the repo and makes every merge painful. The brief also says *raw CSV files remain immutable* — keeping them out of git and read-only makes that a structural fact, not a promise.
* **`nbstripout`.** A notebook is JSON containing code **and outputs** (plots as base64 blobs). Two teammates editing different cells produce unmergeable diffs if outputs are committed. Stripping outputs on commit gives clean, reviewable PR diffs for the "one branch per person + 1 reviewer" rule. The filter is per-clone, hence the "run once per clone" note.
* **One `DATA_DIR` resolver** (next cell). On Kaggle the data lives under `/kaggle/input/<dataset-slug>/`; locally it is wherever each teammate unzipped it. A resolver that *searches* (env var → Kaggle input → common local folders) means the same notebook runs in both places without anyone editing a path — which is exactly the "Run All, no manual fixes" requirement.
* **Shared `src/f1_clean.py`.** The notebook cell for `load_clean()` and the `.py` module are generated from the same source, so P2/P3/P4 can `from f1_clean import load_clean` locally and the Kaggle notebook uses the identical code.

In [ ]:
import os, re, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
SENT = "\\N"          # the literal two-character sentinel used by the Ergast CSVs: backslash + N
KEYS = ["raceId", "driverId"]   # grain of the modeling table
TABLES = ["circuits", "constructors", "constructor_results", "constructor_standings",
          "drivers", "driver_standings", "races", "results", "qualifying", "lap_times",
          "pit_stops", "sprint_results", "seasons", "status"]


def set_all_seeds(seed: int = SEED) -> None:
    """Fix every RNG we might touch. Torch/TF are optional (P4 will use one of them)."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except Exception:
        pass


def _has_data(p: Path) -> bool:
    return p.is_dir() and all((p / f"{t}.csv").exists() for t in TABLES)


def resolve_data_dir() -> Path:
    """Find the folder holding the 14 CSVs, on Kaggle or locally, with no manual edits.
    Order: F1_DATA_DIR env var -> any /kaggle/input/**/races.csv -> common local folders."""
    candidates = []
    if os.environ.get("F1_DATA_DIR"):
        candidates.append(Path(os.environ["F1_DATA_DIR"]))
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates += [p.parent for p in sorted(kaggle_root.rglob("races.csv"))]
    candidates += [Path(p) for p in ("data/raw", "../data/raw", "data", "../data", ".")]
    for p in candidates:
        if _has_data(p):
            return p.resolve()
    raise FileNotFoundError(
        "Could not find the 14 F1 CSVs. On Kaggle: attach the dataset "
        "'rohanrao/formula-1-world-championship-1950-2020' via 'Add Data'. "
        "Locally: put the CSVs in data/raw/ or set the F1_DATA_DIR environment variable.")


def _read_csv_raw(path: Path) -> pd.DataFrame:
    # dtype=str + keep_default_na=False: nothing is guessed, nothing is silently turned into NaN.
    # The raw file is therefore reproduced exactly, including every literal "\N".
    kw = dict(dtype=str, keep_default_na=False, na_values=[])
    try:
        return pd.read_csv(path, encoding="utf-8", **kw)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kw)


def read_raw_tables(data_dir) -> dict:
    return {t: _read_csv_raw(Path(data_dir) / f"{t}.csv") for t in TABLES}

In [ ]:
import time as _time
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
try:                                   # `display` exists in Jupyter; fall back to print elsewhere
    display
except NameError:
    display = print

_T0 = _time.time()
warnings.filterwarnings("ignore", category=FutureWarning)
set_all_seeds(SEED)                    # 1) reproducibility: fixed once, at the very top
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "figure.autolayout": True})

DATA_DIR = resolve_data_dir()          # 2) works on Kaggle and locally
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("DATA_DIR  :", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

---
# 2. Data loading & primary / foreign-key audit

### Why this choice? (loading everything as *strings*)
We read all 14 CSVs with `dtype=str, keep_default_na=False, na_values=[]`.
* The Ergast files use the literal two-character string `\N` for "missing". If we let `read_csv` guess types, any numeric column that contains even one `\N` silently becomes `object`, and pandas' default NA list would hide **other** placeholders (e.g. the text `NA` or `None` as a real value) — we want to *see and count* what the file really contains before deciding anything.
* Loading raw-as-text also guarantees the **raw tables stay immutable**: the dict `RAW` is never modified; every later step works on `.copy()`s.

In [ ]:
RAW = read_raw_tables(DATA_DIR)          # immutable by convention: never assign into RAW
summary = pd.DataFrame({t: {"rows": len(df), "columns": df.shape[1]} for t, df in RAW.items()}).T
display(summary)
assert set(RAW) == set(TABLES) and len(RAW) == 14

### Audit design
**Primary keys.** Each table's PK must be **unique** and **non-null** (a null or the sentinel `\N` counts as null).
Most tables have a surrogate id (`raceId`, `resultId`, …). `lap_times` and `pit_stops` have no surrogate id, so their natural composite key is `(raceId, driverId, lap)` and `(raceId, driverId, stop)`.

**Foreign keys.** For every child column that points to a parent PK we test *referential integrity*: every non-null child value must exist in the parent column. We report the number and share of **orphans**.

### Why this choice? (why the audit is mandatory)
Everything in the milestone is a multi-hop join (results → races → circuits; results → status; results ↔ qualifying; standings → races).
* A **duplicated PK** in a parent makes a join *fan out*: one result row matches two driver rows and silently becomes two → inflated counts, wrong rates, and a broken `(raceId, driverId)` grain.
* A **null PK** can never match anything, so those rows vanish from inner joins (silent sample loss) or become NaN-filled rows in left joins.
* An **orphan FK** (e.g. a result whose `raceId` is not in `races`) is dropped by inner joins and NaN-filled by left joins, which then looks like "missing data" when it is really a broken link.
Auditing first tells us which joins are safe, and where we must choose `how="inner"` vs `"left"` and `validate="m:1"` deliberately.

In [ ]:
PKS = {
    "circuits": ["circuitId"], "constructors": ["constructorId"], "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"], "drivers": ["driverId"],
    "driver_standings": ["driverStandingsId"], "races": ["raceId"], "results": ["resultId"],
    "qualifying": ["qualifyId"], "lap_times": ["raceId", "driverId", "lap"],
    "pit_stops": ["raceId", "driverId", "stop"], "sprint_results": ["resultId"],
    "seasons": ["year"], "status": ["statusId"],
}
FKS = [  # (child table, child column, parent table, parent column)
    ("results", "raceId", "races", "raceId"), ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"), ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"), ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("races", "circuitId", "circuits", "circuitId"), ("races", "year", "seasons", "year"),
    ("sprint_results", "raceId", "races", "raceId"), ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "statusId", "status", "statusId"),
    ("driver_standings", "raceId", "races", "raceId"), ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"),
    ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"),
    ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"), ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"), ("pit_stops", "driverId", "drivers", "driverId"),
]

def is_missing(s: pd.Series) -> pd.Series:
    return s.isna() | s.isin([SENT, ""])

def audit_primary_keys(raw):
    rows = []
    for t, key in PKS.items():
        df = raw[t]
        null_rows = int(is_missing(df[key]).any(axis=1).sum()) if len(key) > 1 else int(is_missing(df[key[0]]).sum())
        dup_rows = int(df.duplicated(key, keep=False).sum())
        rows.append(dict(table=t, primary_key="+".join(key), rows=len(df), null_key_rows=null_rows,
                         rows_in_duplicate_keys=dup_rows, PASS=(null_rows == 0 and dup_rows == 0)))
    return pd.DataFrame(rows)

def audit_foreign_keys(raw):
    rows = []
    for ct, cc, pt, pc in FKS:
        child = raw[ct][cc]
        non_null = child[~is_missing(child)]
        parent_vals = set(raw[pt][pc])
        orphan_mask = ~non_null.isin(parent_vals)
        rows.append(dict(child=f"{ct}.{cc}", parent=f"{pt}.{pc}", child_rows=len(child),
                         null_fk=int(len(child) - len(non_null)), orphans=int(orphan_mask.sum()),
                         orphan_pct=round(100 * orphan_mask.mean(), 4) if len(non_null) else 0.0,
                         example_orphans=list(non_null[orphan_mask].unique()[:3]),
                         PASS=bool(orphan_mask.sum() == 0)))
    return pd.DataFrame(rows)

PK_AUDIT = audit_primary_keys(RAW)
FK_AUDIT = audit_foreign_keys(RAW)
print("PRIMARY KEYS: unique + non-null")
display(PK_AUDIT)
print("FOREIGN KEYS: referential integrity")
display(FK_AUDIT)
print(f"\nPK checks passed: {int(PK_AUDIT.PASS.sum())}/{len(PK_AUDIT)} | FK checks passed: {int(FK_AUDIT.PASS.sum())}/{len(FK_AUDIT)}")

fig, ax = plt.subplots(figsize=(9, 5))
status_df = pd.concat([PK_AUDIT.assign(name=PK_AUDIT.table + " (PK)", problems=PK_AUDIT.null_key_rows + PK_AUDIT.rows_in_duplicate_keys)[["name", "problems"]],
                       FK_AUDIT.assign(name=FK_AUDIT.child + " -> " + FK_AUDIT.parent, problems=FK_AUDIT.orphans + FK_AUDIT.null_fk)[["name", "problems"]]])
bad = status_df[status_df.problems > 0]
if len(bad):
    sns.barplot(data=bad, y="name", x="problems", ax=ax, color="tab:red"); ax.set_title("Keys with problems (rows affected)")
else:
    ax.text(0.5, 0.5, "All PK/FK checks passed", ha="center", va="center", fontsize=14); ax.axis("off")
plt.show()

### How to read / act on the audit
* **PK PASS** → that table can be used as the *one* side of a `m:1` join without fan-out.
* **FK orphans > 0** → we do **not** "repair" by inventing parents. Policy used downstream: results rows whose `raceId` has no race are dropped by the inner join in `load_clean()` (they cannot be placed in time, so they cannot be split train/val/test); orphan `driverId` / `constructorId` / `statusId` would be kept but flagged. The `meta` dict returned by `load_clean()` counts any such rows so nothing disappears silently.
* **Expected nuance (confirm in the table above):** `null_fk` can legitimately be > 0 for nullable FKs (e.g. an unknown `statusId`); that is *missing*, not *broken*. The audit separates the two on purpose.
* `qualifying`, `results` and `sprint_results` PKs are surrogate ids and unique **by construction**; the real integrity question for our grain is `(raceId, driverId)` uniqueness, which a surrogate PK does **not** guarantee — that is Section 5.

---
# 3. Exploratory data analysis — **before** cleaning

We measure the raw tables *as delivered*: how many `\N` sentinels there are, how much is missing, how big the fields are, how rare podiums are, and when qualifying data exists. The same metrics are recomputed after cleaning in Section 7.

### Why this choice? (all EDA is computed, not eyeballed)
The brief requires that EDA is "supported with implementation choices and visualisation". So each plot below is generated from a small, documented function over the raw frames, and the **field-size-aware** metrics (entrants per race, podium rate vs. the 3/N benchmark) are chosen because a podium is always exactly 3 slots, so its base rate is mechanically determined by how many cars take part.

In [ ]:
def num(s: pd.Series) -> pd.Series:
    """raw string column -> numeric (sentinel -> NaN). Used only for EDA on RAW; cleaning is Section 4."""
    return pd.to_numeric(s.replace(SENT, np.nan), errors="coerce")

# ---- (a) sentinel count per table.column -------------------------------------------------------
sent_rows = []
for t, df in RAW.items():
    for c in df.columns:
        n = int((df[c] == SENT).sum())
        sent_rows.append(dict(table=t, column=c, sentinels=n, rows=len(df), pct=100 * n / max(len(df), 1)))
SENT_COUNTS = pd.DataFrame(sent_rows)
has = SENT_COUNTS[SENT_COUNTS.sentinels > 0].sort_values("pct", ascending=False)
print(f"{len(has)} of {len(SENT_COUNTS)} columns contain at least one literal \\N")
display(has.head(40).reset_index(drop=True))

top = has.head(25).assign(label=lambda d: d.table + "." + d.column)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=top, y="label", x="pct", ax=axes[0], color="tab:blue")
axes[0].set_title("Top-25 columns by share of literal \\N"); axes[0].set_xlabel("% of rows = \\N"); axes[0].set_ylabel("")
# ---- (b) overall missingness distribution ------------------------------------------------------
sns.histplot(SENT_COUNTS.pct, bins=20, ax=axes[1], color="tab:orange")
axes[1].set_yscale("log"); axes[1].set_xlabel("% missing (\\N) per column"); axes[1].set_ylabel("# columns (log)")
axes[1].set_title(f"Distribution of missingness over all {len(SENT_COUNTS)} columns")
plt.show()

per_table = SENT_COUNTS.groupby("table").apply(lambda g: pd.Series({"cols": len(g), "cols_with_\\N": int((g.sentinels > 0).sum()),
                                      "cells_\\N": int(g.sentinels.sum()), "pct_cells_\\N": 100 * g.sentinels.sum() / (g.rows.iloc[0] * len(g))}),
                                      include_groups=False).sort_values("pct_cells_\\N", ascending=False)
display(per_table.round(2))

**What this shows.** Missingness is *concentrated*, not uniform: most columns have no sentinel at all, and a handful carry most of it. Typically (confirm in the plot) these are: qualifying sessions `q2`/`q3` (cars eliminated in Q1/Q2 never set a Q2/Q3 time — that missingness is *structural*, not random), `fastestLapTime`/`rank`/`fastestLapSpeed` in old results (timing not recorded), `driver.number`, and the newer `races` schedule columns (`fp1_date`…`sprint_date`, which only exist for recent seasons).

**Why this matters for cleaning:** the right treatment differs by cause — structural (eliminated in Q1 ⇒ no `q3`), historical (not recorded in the 1950s), or genuine absence (no sprint that weekend). So Section 4 converts sentinels to `NaN` **without imputing**; any imputation happens later, on the training split only.

In [ ]:
# ---- (c) entrants per race & (d) base podium rate per season (RAW, duplicates still inside) -----
def season_stats(df):
    """df needs: year, raceId, driverId, positionOrder -> one row per season."""
    df = df.assign(_pod=(df["podium"].astype(bool) if "podium" in df else (df["positionOrder"] <= 3)))
    g = df.groupby("year")
    out = pd.DataFrame({"races": g["raceId"].nunique(), "rows": g.size(),
                        "unique_drivers_per_race": g.apply(lambda x: x.groupby("raceId")["driverId"].nunique().mean(), include_groups=False),
                        "podium_rows": g["_pod"].sum().astype(int)})
    out["mean_field"] = out.rows / out.races
    out["podium_rate"] = out.podium_rows / out.rows
    out["benchmark_3_over_N"] = 3 / out.mean_field          # what a 3-slot podium implies for field size N
    return out

_r = RAW["results"][["raceId", "driverId", "positionOrder"]].assign(positionOrder=lambda d: num(d.positionOrder))
_races = RAW["races"][["raceId", "year"]].assign(year=lambda d: num(d.year))
RAW_RES = _r.merge(_races, on="raceId", how="inner")
SEASON_RAW = season_stats(RAW_RES)

per_race = RAW_RES.groupby(["year", "raceId"]).size().rename("entrants").reset_index()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
agg = per_race.groupby("year").entrants.agg(["min", "mean", "max"])
axes[0].fill_between(agg.index, agg["min"], agg["max"], alpha=.25, label="min-max across races")
axes[0].plot(agg.index, agg["mean"], lw=2, label="mean entrants / race (raw rows)")
axes[0].plot(SEASON_RAW.index, SEASON_RAW.unique_drivers_per_race, "--", lw=1.5, color="k", label="unique drivers / race")
axes[0].set_title("Entrants per race by season (raw)"); axes[0].set_xlabel("season"); axes[0].legend()
ax2 = axes[1]
ax2.plot(SEASON_RAW.index, SEASON_RAW.podium_rate, lw=2, label="observed podium rate (rows with positionOrder<=3)")
ax2.plot(SEASON_RAW.index, SEASON_RAW.benchmark_3_over_N, "--", lw=1.5, label="benchmark 3 / mean field size")
ax2.set_title("Base podium rate per season (raw)"); ax2.set_xlabel("season"); ax2.set_ylabel("share of rows"); ax2.legend()
plt.show()
print(f"Overall raw podium rate: {(RAW_RES.positionOrder <= 3).mean():.3%}  | seasons: {SEASON_RAW.index.min()}-{SEASON_RAW.index.max()}")
display(SEASON_RAW.iloc[[0, 5, 10, 20, 30, 40, 50, 60, 70, -1]].round(3))

**What these two plots reveal**
* **Field size is not constant.** Fields were large and uneven in the 1950s, peaked in the 1980s–early 1990s (many more cars than grid places), then shrank to ~20 as every team entered exactly two cars. Raw rows per race exceed the number of *unique drivers* in the early decades — the first visible symptom of the duplicate (shared-drive) rows handled in Section 5.
* **Podium scarcity is era-dependent.** The podium is always 3 slots, so the base rate ≈ 3 / field size: roughly 15% with ~20 cars, far lower with 30+. The dashed benchmark tracks the observed rate closely; the gap (where the observed rate is *higher* than the benchmark) again points to rows that do not correspond to distinct drivers.
* **Modeling implication:** a pooled "accuracy" is meaningless, class balance differs by era, and our time split (train ≤ 2019 / val 2020-21 / test ≥ 2022) will have different positive rates in each part — to be reported by P3. This is why PR-AUC, not accuracy, is the natural primary metric.

In [ ]:
# ---- (e) qualifying coverage per season (RAW) ----------------------------------------------------
Q = RAW["qualifying"][["raceId", "driverId", "q1", "q2", "q3"]]
q_keys = Q[["raceId", "driverId"]].drop_duplicates().assign(has_q_row=True)
cov = (RAW["results"][["raceId", "driverId"]].merge(_races, on="raceId")
       .merge(q_keys, on=["raceId", "driverId"], how="left")
       .merge(Q.assign(has_q1=Q.q1.ne(SENT), has_q2=Q.q2.ne(SENT), has_q3=Q.q3.ne(SENT)).groupby(["raceId", "driverId"])[["has_q1", "has_q2", "has_q3"]].max().reset_index(),
              on=["raceId", "driverId"], how="left"))
cov[["has_q_row", "has_q1", "has_q2", "has_q3"]] = cov[["has_q_row", "has_q1", "has_q2", "has_q3"]].fillna(False).astype(bool)
race_has_q = cov.groupby(["year", "raceId"]).has_q_row.any().groupby("year").mean()
COV = cov.groupby("year")[["has_q_row", "has_q1", "has_q2", "has_q3"]].mean().assign(races_with_any_qualifying=race_has_q)

fig, ax = plt.subplots(figsize=(11, 4.8))
for c, lab in [("races_with_any_qualifying", "share of races with ANY qualifying rows"), ("has_q_row", "share of result rows with a qualifying row"),
               ("has_q1", "... with a Q1 time"), ("has_q3", "... with a Q3 time")]:
    ax.plot(COV.index, COV[c], lw=2, label=lab)
first_q = int(COV.index[COV.races_with_any_qualifying > 0].min())
ax.axvline(first_q, color="k", ls=":"); ax.text(first_q + .3, .5, f"first qualifying data: {first_q}")
ax.set_ylim(-.02, 1.02); ax.set_xlabel("season"); ax.set_ylabel("coverage"); ax.set_title("Qualifying coverage per season (raw)"); ax.legend(loc="center left")
plt.show()
print("First season with any qualifying rows:", first_q)
display(COV.loc[first_q - 2:first_q + 12].round(3))

**What this reveals — historical qualifying-coverage drift.**
* Before the first qualifying season printed above there is **no** qualifying table row for any driver; coverage then ramps up (partial seasons, then complete). Q1 is the first session to appear; **Q2 and Q3 only exist for the knockout era (2006+)** — before that a driver can have a "qualifying" row with only `q1`.
* **Why we cannot naively impute early decades.** For 1950–mid-1990s the value is not *missing at random*: the session simply is not in the data, and the timing formats were different (multiple sessions, best-of, pre-qualifying), so a 1962 "Q1 time" would not be comparable to a 2015 one even if we had it. Mean/median imputation would invent a continuous feature for ~half of history from nothing. Options downstream (P3's call, informed by this plot): (i) fall back to `grid` (always present), (ii) add a `has_qualifying` indicator, (iii) restrict the training window to the qualifying era. `load_clean()` therefore ships `has_qualifying` and leaves the time columns as `NaN`.
* Grid position (not qualifying time) is our only continuous starting-order record for most of history, which is why the dataset keeps `grid` as a first-class pre-race column.

---
# 4. Data cleaning & semantic typing

### 4.1 Sentinels → `NaN`, then explicit types
**Why this choice?**
* A cell holding the text `\N` is a *string*. In a numeric column it makes the whole column `object`: `mean()` fails or silently skips, comparisons like `grid <= 2` raise or compare strings lexicographically (`"10" < "2"`), and `sort_values` orders text. In a date column `\N` is not parseable, so `to_datetime` either errors or gives garbage. Replacing it with a true missing value (`NaN`/`NaT`/`<NA>`) is what lets pandas, scikit-learn and SHAP treat "unknown" as unknown.
* **Blank cells count as missing too.** The real CSVs can contain empty strings in addition to `\N`; both mean "no value", so `clean_table` converts both. (The Section-7 check showed a few dozen q2/q3 cells that were neither `\N` nor parseable — this is the fix, and Section 7 now lists any value that is still unparsable.)
* **We convert, we do not impute.** The reason a value is missing differs by column (Section 3), so imputing here would bake unjustified assumptions into the base tables *for every teammate*, and could leak (e.g. imputing with a global mean that includes test-era rows). Imputation belongs in P3's train-only preprocessing pipeline.
* **Typing is explicit** (a dict per table, not `read_csv` guessing): ids and counts become nullable `Int64` (keeps `NaN` without turning ints into floats), points/speeds/coordinates `float64`, dates `datetime64`, lap times seconds as `float64`.
* **Lap times → seconds.** `"1:27.452"` is text; seconds (`87.452`) are comparable, differentiable and usable for `gap to pole (%)`. The parser supports `m:ss.fff`, `ss.fff` and `h:mm:ss.fff` and returns `NaN` for anything else — failures are *counted* in the log below instead of being hidden.

In [ ]:
# ---- Semantic typing specification (explicit > guessed) -------------------------------------
INT_COLS = {
    "races": ["year", "round"], "seasons": ["year"], "drivers": ["number"],
    "constructor_standings": ["position", "wins"], "driver_standings": ["position", "wins"],
    "results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap", "rank"],
    "qualifying": ["number", "position"],
    "lap_times": ["lap", "position", "milliseconds"],
    "pit_stops": ["stop", "lap", "milliseconds"],
    "sprint_results": ["number", "grid", "position", "positionOrder", "laps", "milliseconds", "fastestLap"],
}
FLOAT_COLS = {
    "circuits": ["lat", "lng", "alt"],
    "constructor_results": ["points"], "constructor_standings": ["points"], "driver_standings": ["points"],
    "results": ["points", "fastestLapSpeed"], "sprint_results": ["points", "fastestLapSpeed"],
}
DATE_COLS = {
    "races": ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"],
    "drivers": ["dob"],
}
LAPTIME_COLS = {   # table -> {source string column: new numeric-seconds column}
    "results": {"fastestLapTime": "fastestLapTime_s"},
    "sprint_results": {"fastestLapTime": "fastestLapTime_s"},
    "qualifying": {"q1": "q1_s", "q2": "q2_s", "q3": "q3_s"},
    "lap_times": {"time": "time_s"},
    "pit_stops": {"duration": "duration_s"},
}

_LAP_RE = r"^(?:(?:(\d+):)?(\d+):)?(\d+(?:\.\d+)?)$"   # [h:][m:]s[.fff]


def laptime_to_seconds(s: pd.Series) -> pd.Series:
    """'1:27.452' -> 87.452 ; '59.3' -> 59.3 ; '1:02:03.5' -> 3723.5 ; NaN/garbage -> NaN."""
    parts = s.astype("string").str.extract(_LAP_RE)
    h = pd.to_numeric(parts[0]).fillna(0)
    m = pd.to_numeric(parts[1]).fillna(0)
    sec = pd.to_numeric(parts[2])
    return (h * 3600 + m * 60 + sec).astype("float64")


def to_int(s: pd.Series) -> pd.Series:
    n = pd.to_numeric(s, errors="coerce")
    if (n.notna() & (n % 1 != 0)).any():
        warnings.warn(f"non-integer values rounded in column {s.name}")
    return n.round().astype("Int64")      # nullable Int64 keeps NaN without turning ints into floats


def clean_table(name: str, df: pd.DataFrame):
    """Return (cleaned copy, log rows). The raw frame is never modified."""
    d = df.copy().replace(SENT, np.nan)
    d = d.replace(r"^\s*$", np.nan, regex=True)       # blank / whitespace-only cells are also "no value"
    log = []
    for c in d.columns:                                    # all *Id columns -> nullable ints
        if c.endswith("Id"):
            d[c] = to_int(d[c])
    for c in INT_COLS.get(name, []):
        if c in d: d[c] = to_int(d[c])
    for c in FLOAT_COLS.get(name, []):
        if c in d: d[c] = pd.to_numeric(d[c], errors="coerce").astype("float64")
    for c in DATE_COLS.get(name, []):
        if c in d:
            before = d[c].notna().sum()
            d[c] = pd.to_datetime(d[c], format="%Y-%m-%d", errors="coerce")
            log.append(dict(table=name, column=c, op="to_datetime", non_null_before=int(before),
                            non_null_after=int(d[c].notna().sum())))
    for src, dst in LAPTIME_COLS.get(name, {}).items():
        if src in d:
            before = d[src].notna().sum()
            d[dst] = laptime_to_seconds(d[src])
            log.append(dict(table=name, column=f"{src}->{dst}", op="laptime_to_seconds",
                            non_null_before=int(before), non_null_after=int(d[dst].notna().sum())))
            d = d.drop(columns=[src])           # keep one canonical column: numeric seconds
    if name == "races" and {"date", "time"} <= set(d.columns):   # scheduled start, known pre-race
        stamp = d["date"].dt.strftime("%Y-%m-%d") + " " + d["time"].astype("string")
        d["race_start_utc"] = pd.to_datetime(stamp, errors="coerce")
    return d, log


def clean_all(raw: dict):
    out, logs = {}, []
    for name, df in raw.items():
        out[name], lg = clean_table(name, df)
        logs += lg
    return out, pd.DataFrame(logs)

In [ ]:
CLEAN, CLEAN_LOG = clean_all(RAW)        # works on copies; RAW is untouched
display(CLEAN_LOG)
print("lap-time/date values that were present but could not be parsed (expected ~0; inspect below if not):", int((CLEAN_LOG.non_null_before - CLEAN_LOG.non_null_after).sum()))

def count_sentinels(df):
    return int(sum((df[c] == SENT).sum() for c in df.columns
                   if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])))
left_over = {t: count_sentinels(df) for t, df in CLEAN.items()}
assert sum(left_over.values()) == 0, left_over
print("literal \\N remaining in cleaned tables:", sum(left_over.values()))

# unit test of the lap-time parser
tests = pd.Series(["1:27.452", "59.3", "0:59.300", "1:02:03.5", np.nan, "garbage"])
print(dict(zip(tests.fillna("NaN"), laptime_to_seconds(tests).round(3))))
assert abs(laptime_to_seconds(pd.Series(["1:27.452"])).iloc[0] - 87.452) < 1e-9

_pairs = [("results", "grid", "grid"), ("results", "positionOrder", "positionOrder"), ("results", "points", "points"),
          ("results", "fastestLapTime", "fastestLapTime_s"), ("qualifying", "q3", "q3_s"),
          ("races", "date", "date"), ("drivers", "dob", "dob")]
cmp = pd.DataFrame([(t, f"{r} -> {c}", str(RAW[t][r].dtype), str(CLEAN[t][c].dtype)) for t, r, c in _pairs],
                   columns=["table", "column", "dtype before", "dtype after"])
display(cmp)

# Typed results joined to what we need for the rest of the analysis
RES_ALL = (CLEAN["results"]
           .merge(CLEAN["races"][["raceId", "year", "round", "date", "circuitId"]], on="raceId", how="inner", validate="m:1")
           .merge(CLEAN["status"], on="statusId", how="left", validate="m:1"))
print("typed results rows:", len(RES_ALL), "| raw result rows:", len(RAW["results"]))

### 4.2 What does `grid = 0` mean?
The project primer says `0` is a pit-lane start *"(verify in EDA)"*. We test two hypotheses rather than assume:
* **H1 (pit-lane start):** the driver did race (laps > 0, a real finishing status), just without a grid slot — mostly modern years.
* **H2 (no start / no slot):** the row is a DNQ / withdrawn / non-starter, so `0` means "no grid position" — mostly historical.

Evidence: share of `grid = 0` per decade split by *did the driver complete laps*, the `positionText` codes, the status values, and (1996+) whether the driver has a qualifying time (a driver who **qualified** but has `grid=0` was moved to the pit lane by a penalty/set-up change).

In [ ]:
G = RES_ALL.copy()
G["is0"] = G["grid"].eq(0).fillna(False).astype(bool)
G["decade"] = (G["year"] // 10 * 10).astype(int)
G["ran"] = G["laps"].fillna(0) > 0
tab = G.groupby("decade").apply(lambda d: pd.Series({
    "rows": len(d), "grid0": int(d.is0.sum()), "share_grid0": d.is0.mean(),
    "grid0_and_ran": int((d.is0 & d.ran).sum()), "grid0_no_laps": int((d.is0 & ~d.ran).sum()),
    "grid0_positionText_F_or_W": int((d.is0 & d.positionText.isin(["F", "W"])).sum())}), include_groups=False)
display(tab.round(3))
print("grid is NaN:", int(G.grid.isna().sum()), "| grid < 0:", int((G.grid < 0).sum()))

fig, ax = plt.subplots(figsize=(10, 4.5))
tab[["grid0_and_ran", "grid0_no_laps"]].plot(kind="bar", stacked=True, ax=ax, color=["tab:green", "tab:red"])
ax.set_title("Rows with grid = 0 by decade: did the driver complete laps?"); ax.set_ylabel("rows"); ax.set_xlabel("decade")
plt.show()

for lab, d in [("pre-1990 grid=0", G[G.is0 & (G.year < 1990)]), ("2000+ grid=0", G[G.is0 & (G.year >= 2000)])]:
    print(f"\n{lab}: n={len(d)} | share that completed laps: {d.ran.mean():.1%}")
    print("  positionText:", d.positionText.value_counts().head(5).to_dict())
    print("  top statuses:", d.status.value_counts().head(6).to_dict())

Qc = CLEAN["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "qpos"})
m = G[G.is0 & (G.year >= 1996)].merge(Qc, on=["raceId", "driverId"], how="left")
if len(m):
    print(f"\n1996+ grid=0 rows: {len(m)} | with a qualifying row: {m.qpos.notna().mean():.1%} | of those who ran: {(m.ran).mean():.1%}")

### Conclusion on `grid = 0` (confirm against the tables above)
`grid = 0` is **not** one thing:
1. **Modern era (pit-lane start):** the car *did* race (laps > 0, normal finishing statuses). In F1 sporting regulations a driver who breaks parc-fermé, changes major parts after qualifying, or cannot reach the grid in time starts from the **pit lane** after the field has left — so there is no grid slot, effectively "behind everyone". Many of these drivers *do* have a qualifying time, i.e. they qualified and were **moved**.
2. **Historical era / no-start rows:** `0` also marks drivers with *no grid slot at all* — DNQ (`positionText = F`), withdrawn (`W`) and non-starters — plus a minority of starters whose grid slot was simply not recorded.

**Why this choice for our table?** `0` must never be fed to a model as "pole position −1" (it would look like the best grid slot). We (a) keep the raw `grid`, (b) **remove non-starters first** (Section 6), then (c) define `pitlane_start = (grid == 0)` among the *remaining* rows, so the flag means what it says; P3 can then map those rows to "last on the grid" or keep the dummy flag. This is also pre-race information: penalties and pit-lane starts are published before lights-out.

---
# 5. Grain enforcement & duplicate handling

**Required grain:** *exactly one row per driver per race* — `(raceId, driverId)` must be unique, because every row is one prediction ("will **this driver** finish on the podium in **this race**?").

### Why this choice? (why the grain is strict)
* **Binary classification needs one label per prediction unit.** Two rows for the same driver-race can carry different labels (one podium, one not) and different features (two cars ⇒ two `constructorId`s, two `grid`s) — the model would be asked to predict contradictory outcomes for the same entity.
* **Leakage across the split.** Duplicates are not independent samples; the same race could appear in train *and* validation through its twin row (with a random split) and inflate scores. A unique key makes time-split assertions trivially checkable.
* **Wrong rates everywhere downstream.** Podium rate, mechanical-retirement rate (Q3), home-race rate (Q2) all use rows as the denominator.
* **Why 1950s duplicates exist.** Until 1957 a team could call a driver in mid-race and hand his car to a teammate (or a star could take over a teammate's car after his own broke). Points were split. The dataset stores one result row **per car**, so the same driver gets two rows in one race.

In [ ]:
dup_mask = RES_ALL.duplicated(KEYS, keep=False)
excess = RES_ALL.duplicated(KEYS, keep="first")
years = pd.RangeIndex(1950, 2025)
BY_YEAR = pd.DataFrame({"rows_in_dup_groups": dup_mask.groupby(RES_ALL.year).sum(),
                        "excess_rows_to_drop": excess.groupby(RES_ALL.year).sum()}).reindex(years, fill_value=0)
print(f"rows in duplicate (raceId, driverId) groups: {int(dup_mask.sum())} | groups: {RES_ALL[dup_mask].groupby(KEYS).ngroups} | excess rows: {int(excess.sum())}")
print(f"share of excess rows in 1950-1957: {BY_YEAR.loc[1950:1957, 'excess_rows_to_drop'].sum() / max(BY_YEAR.excess_rows_to_drop.sum(), 1):.1%}  | last season with any duplicate: "
      f"{int(BY_YEAR.index[BY_YEAR.excess_rows_to_drop > 0].max()) if (BY_YEAR.excess_rows_to_drop > 0).any() else 'none'}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(BY_YEAR.index, BY_YEAR.excess_rows_to_drop, color="tab:red")
ax.set_title("Duplicate (raceId, driverId) rows per season (excess rows = rows beyond the first)"); ax.set_xlabel("season"); ax.set_ylabel("excess rows")
plt.show()

ex = (RES_ALL[dup_mask].merge(CLEAN["drivers"][["driverId", "surname"]], on="driverId")
      .merge(CLEAN["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId", how="left"))
display(ex.sort_values(["year", "raceId", "surname", "resultId"])[["year", "raceId", "surname", "constructor", "number", "grid", "positionOrder", "points", "status"]].head(12))

### The keep-rule — and the "no post-race information" principle
Choosing a row involves **two different jobs**, and they follow different rules:

| job | may it use post-race information? |
|---|---|
| **A. Choose which row supplies the pre-race columns** (`grid`, `constructorId`, car `number`) | **No.** Features must be known at the cutoff; if the *outcome* decides which features a driver gets, the outcome leaks into the features. |
| **B. Define the label `podium`** | **Yes.** The label *is* the outcome; it is the one place post-race data belongs. |

**Job A — row choice uses pre-race facts only** (total order, first wins):

| priority | criterion | why |
|---|---|---|
| 1 | a row with **`grid > 0`** | a car that actually started; known at the cutoff |
| 2 | **lowest grid slot** | when several started cars are candidates, a pre-race convention. Pre-race data cannot tell *which* of two started cars the driver "owned"; we pick the best slot and flag the row |
| 3 | `constructorId`, then car `number` | arbitrary but outcome-free tie-breaks |
| 4 | `resultId` | last resort. In this dataset result ids are assigned in finishing order, so it is only reached when the rows are identical in every pre-race column — it cannot change any feature |

`positionOrder` is **not** used to choose the row. This is *verified in code*: `check_dedup_is_outcome_free` scrambles all outcome columns and asserts that exactly the same rows are kept.

**Honest limitation of Job A.** For a shared drive in which several candidate cars started, the kept `grid` may be the teammate's slot rather than the driver's own — the pre-race columns simply do not contain enough information to know. This is small and bounded (shared drives are ≈0.3% of rows, all ≤ 1957 — none in validation/test); every affected row is flagged (`shared_drive`), and `load_clean(exclude_shared_drives=True)` removes them if the team prefers zero ambiguity.

**Job B — the label.** `podium = 1` if the driver's **best `positionOrder` across all his rows in the race is ≤ 3**. A driver who took over a car that finished on the podium was credited with that finish (points were split), so he did finish on the podium. For every non-shared row this is simply `positionOrder ≤ 3`. The label may use post-race data because it is the *target*, never a feature.

**Why keep one row rather than drop all shared drives?** Dropping them loses real podium finishes and (race, driver) coverage; keeping one row preserves every driver-race pair. **Logical guarantee:** the sort key is a total order, so `drop_duplicates(keep="first")` returns exactly one deterministic row per key — asserted below.

In [ ]:
# ---- Population rules ------------------------------------------------------------------------
INDY_YEARS = (1950, 1960)
NONSTART_POSITION_TEXT = {"F", "W"}          # F = failed to qualify, W = withdrawn
NONSTART_STATUS_RE = r"did not (?:pre)?qualify|not qualified|withdrew|107%"


def flag_races(races: pd.DataFrame, circuits: pd.DataFrame) -> pd.DataFrame:
    """Add is_indy500: the 1950-1960 Indianapolis 500 rounds that counted for the F1 championship.
    NOTE: the same Indianapolis circuit hosted a real F1 US GP in 2000-2007 -> keep those (year filter)."""
    r = races.copy()
    indy_ids = circuits.loc[
        circuits["name"].str.contains("Indianapolis", case=False, na=False)
        | circuits["circuitRef"].str.contains("indianapolis", case=False, na=False), "circuitId"]
    by_name = r["name"].str.contains("Indianapolis 500", case=False, na=False)
    by_venue = r["circuitId"].isin(indy_ids) & r["year"].between(*INDY_YEARS)
    r["is_indy500"] = (by_name | by_venue).astype(bool)
    return r


def flag_non_starters(res: pd.DataFrame) -> pd.Series:
    """DNQ / DNPQ / withdrawn / 107%-rule rows. Uses only facts that are settled BEFORE lights-out
    (who failed to qualify or withdrew) - never laps or finishing status of drivers who did race."""
    by_text = res["positionText"].isin(NONSTART_POSITION_TEXT)
    by_status = res["status"].str.contains(NONSTART_STATUS_RE, case=False, regex=True, na=False)
    return (by_text | by_status).astype(bool)


def dedup_results(res: pd.DataFrame):
    """Enforce exactly one row per (raceId, driverId).

    Two separate jobs, with different rules about post-race information:
      (1) WHICH row supplies the pre-race columns (grid, constructorId, number): may use ONLY pre-race facts.
          Total order, first row wins:
            a. a row with grid > 0        (a car that actually started)
            b. lowest grid slot            (pre-race convention when several started cars are candidates)
            c. constructorId, then number  (arbitrary but outcome-free tie-breaks)
            d. resultId                    (last resort: reached only if all pre-race columns are identical,
                                            so it cannot change any feature)
          positionOrder is NOT used here, and neither is anything correlated with it (resultId is assigned in
          finishing order in this dataset, hence it is only a last resort).
      (2) The LABEL `podium`: the target IS the outcome, so it may use post-race data. A driver who shared a
          car and was credited with a podium finish in either car did finish on the podium:
          podium = (best positionOrder over ALL the driver's rows in the race) <= 3.
    Returns (kept rows with audit columns, dropped rows)."""
    d = res.reset_index(drop=True).copy()
    d["n_raw_rows"] = d.groupby(KEYS)["resultId"].transform("size").astype("int64")
    d["best_positionOrder_raw"] = d.groupby(KEYS)["positionOrder"].transform("min")   # used ONLY for the label
    d["_started"] = d["grid"].fillna(0).gt(0).astype(bool)
    d["_grid_key"] = d["grid"].where(d["_started"])                 # NaN (sorted last) when no grid slot
    d = d.sort_values(KEYS + ["_started", "_grid_key", "constructorId", "number", "resultId"],
                      ascending=[True, True, False, True, True, True, True], kind="mergesort")
    kept = d.drop_duplicates(KEYS, keep="first")
    dropped = d.loc[~d.index.isin(kept.index)].drop(columns=["_started", "_grid_key"])
    kept = kept.drop(columns=["_started", "_grid_key"])
    kept = kept.assign(shared_drive=kept["n_raw_rows"] > 1,
                       podium=(kept["best_positionOrder_raw"] <= 3).astype("int8"))
    kept = kept.sort_values(["raceId", "positionOrder", "driverId"]).reset_index(drop=True)
    return kept, dropped


def check_dedup_is_outcome_free(res: pd.DataFrame, seed: int = SEED) -> None:
    """Executable proof of 'no post-race info in the row choice': scramble every outcome column across rows
    and verify that the SAME rows are kept. If a future edit lets positionOrder influence the choice,
    this assertion fails."""
    rng = np.random.default_rng(seed)
    shuffled = res.copy()
    for c in ("positionOrder", "position", "points", "laps", "statusId"):
        if c in shuffled:
            shuffled[c] = rng.permutation(shuffled[c].to_numpy())
    kept_a, _ = dedup_results(res)
    kept_b, _ = dedup_results(shuffled)
    assert set(kept_a["resultId"]) == set(kept_b["resultId"]), "row choice depends on post-race columns"

In [ ]:
RES_DEDUP, DROPPED = dedup_results(RES_ALL)
assert not RES_DEDUP.duplicated(KEYS).any()
assert len(RES_DEDUP) + len(DROPPED) == len(RES_ALL)
assert RES_DEDUP[KEYS].drop_duplicates().shape[0] == RES_ALL[KEYS].drop_duplicates().shape[0]   # no (race, driver) pair lost
print(f"rows: {len(RES_ALL)} -> {len(RES_DEDUP)} (dropped {len(DROPPED)}) | every (raceId, driverId) pair preserved")

check_dedup_is_outcome_free(RES_ALL)       # proves positionOrder & co. play no role in WHICH row is kept
print("outcome-independence check passed: scrambling every outcome column keeps exactly the same rows")

# How ambiguous was each duplicate group, judged ONLY with pre-race columns?
g = RES_ALL[dup_mask].copy()
g["started"] = g.grid.fillna(0).gt(0).astype(bool)
g["sig"] = list(zip(g.constructorId.astype("int64"), g.grid.fillna(0).astype("int64"), g.number.fillna(-1).astype("int64")))
def classify(x):
    pool = x[x["started"]] if x["started"].any() else x
    if x["started"].sum() == 1:
        return "A: exactly one started row -> unambiguous"
    if pool["sig"].nunique() == 1:
        return "B: several candidates, identical pre-race columns -> choice cannot change any feature"
    return "C: several candidates with DIFFERENT pre-race columns -> lowest-grid convention decides"
tiers = g.groupby(KEYS).apply(classify, include_groups=False)
display(tiers.value_counts().rename("duplicate groups").to_frame())

# Label impact: label from the single kept row vs the driver's best row (the label we use)
sd = RES_DEDUP[RES_DEDUP.shared_drive]
row_label = (sd.positionOrder <= 3).astype(int)
print(f"shared-drive kept rows: {len(sd)} ({len(sd) / len(RES_DEDUP):.3%} of rows)")
display(pd.DataFrame({"podium=1 if label came from the kept row only": [int(row_label.sum())],
                      "podium=1 with label = driver's best row (used)": [int(sd.podium.sum())],
                      "labels that differ": [int((row_label != sd.podium).sum())],
                      "share of ALL rows affected": [f"{(row_label != sd.podium).sum() / len(RES_DEDUP):.4%}"]}))

In [ ]:
# Quantify the residual risk of the pre-race convention (lowest grid) for shared drives
mrg = DROPPED.merge(RES_DEDUP[KEYS + ["constructorId", "grid", "year"]], on=KEYS, suffixes=("_dropped", "_kept"))
print(f"seasons containing shared drives: {int(sd.year.min())}-{int(sd.year.max())}  (validation 2020-21 / test >=2022 affected: {int((sd.year >= 2020).sum())} rows)")
print(f"dropped rows whose constructorId differs from the kept row: {int((mrg.constructorId_dropped != mrg.constructorId_kept).sum())} of {len(mrg)}"
      f"  | whose grid differs: {int((mrg.grid_dropped.fillna(-1) != mrg.grid_kept.fillna(-1)).sum())} of {len(mrg)}")
print(f"podium share among shared-drive rows: {sd.podium.mean():.1%} vs all rows: {RES_DEDUP.podium.mean():.1%}")

**Reading the outputs above.**
* **Group types:** type A groups are unambiguous; type B groups cannot affect any feature; only type C groups depend on the lowest-grid convention, and the *count* of those is the real size of the issue (and `exclude_shared_drives=True` removes all of A/B/C if wanted).
* **Labels:** the label difference table shows how many rows would be labelled differently if we naively took the label from the single kept row. Using the driver's best row is the faithful definition of "this driver finished on the podium".
* **Era:** all shared drives are early 1950s, so there is **no** effect on validation/test.
* **Why this answers the leakage question:** the *features* never depend on the outcome (executable check above); the outcome appears only in the *label*, where it belongs.

---
# 6. Early-F1 filtering: Indianapolis 500 (1950–1960) and DNQ / non-starters

### 6.1 The Indianapolis 500
The Indy 500 counted as a round of the World Championship in 1950–1960 but almost no F1 regulars entered it, and its entrants rarely raced elsewhere.

**Decision: exclude those races** from the modeling table (flag `is_indy500` stays on `races` so P2 can still see them).

**Why this choice?**
* **Different sport.** A 33-car oval field with a very different grid structure (rows of 3, no pit-lane / DNQ logic like a road course) — grid ↔ podium conversion there is not the same phenomenon as at an F1 circuit.
* **Distorts nationality / home-race analysis (Q2).** Nearly all Indy entrants were American, so "American driver at a US circuit" would be dominated by Indy rows with no relation to F1 form.
* **Distorts form features.** Rolling driver form built from Indy results would mix a handful of one-off drivers into a history that never continues into F1.
* **Caution:** the same Indianapolis *circuit* hosted a genuine F1 US Grand Prix in 2000–2007. We therefore identify the 500 by **race name and year ≤ 1960**, not by circuit alone.

In [ ]:
RACES = flag_races(CLEAN["races"], CLEAN["circuits"])
indy_races = RACES[RACES.is_indy500]
print(f"Indy 500 championship rounds flagged: {len(indy_races)} | seasons: {sorted(indy_races.year.unique().tolist())}")
assert indy_races.year.between(1950, 1960).all()
assert RACES[(RACES.year.between(2000, 2007)) & RACES.is_indy500].empty, "modern US GP at Indianapolis must NOT be flagged"

RES_DEDUP = RES_DEDUP.merge(RACES[["raceId", "is_indy500"]], on="raceId", how="left", validate="m:1")
tmp = RES_DEDUP.merge(CLEAN["drivers"][["driverId", "nationality"]], on="driverId", how="left")
tmp["american"] = tmp.nationality.eq("American")
tmp["us_circuit"] = tmp.circuitId.isin(CLEAN["circuits"].loc[CLEAN["circuits"].country.isin(["USA", "United States"]), "circuitId"])
p = tmp[tmp.year.between(1950, 1960)].groupby(["year", "is_indy500"]).american.sum().unstack(fill_value=0)
p.columns = ["American driver-entries at F1 races" if not c else "American driver-entries at Indy 500" for c in p.columns]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
p.plot(kind="bar", stacked=True, ax=axes[0], color=["tab:blue", "tab:red"]); axes[0].set_title("American driver-entries per season, 1950-60"); axes[0].set_xlabel("season")
fld = tmp[tmp.year.between(1950, 1960)].groupby(["raceId", "is_indy500"]).size().rename("entrants").reset_index()
sns.boxplot(data=fld, x="is_indy500", y="entrants", ax=axes[1]); axes[1].set_xticks(axes[1].get_xticks()); axes[1].set_xticklabels(["other F1 races", "Indy 500"][:len(axes[1].get_xticks())]); axes[1].set_xlabel(""); axes[1].set_title("Field size, 1950-60")
plt.show()

indy_rows = tmp[tmp.is_indy500]
only_indy = tmp.groupby("driverId").is_indy500.all()
home_us = tmp[tmp.american & tmp.us_circuit]
print(f"Indy rows: {len(indy_rows)} | drivers who ONLY ever appear at the Indy 500: {int(only_indy.sum())} of {tmp.driverId.nunique()}")
print(f"'American driver at US circuit' rows (naive home-race rows): {len(home_us)} -> {int(home_us.is_indy500.sum())} ({home_us.is_indy500.mean():.1%}) are Indy 500 rows")
print(f"podium rate 1950-60: Indy {indy_rows.podium.mean():.3%} vs other F1 {tmp[tmp.year.between(1950, 1960) & ~tmp.is_indy500].podium.mean():.3%}")
dup_share = RES_ALL.merge(RACES[["raceId", "is_indy500"]], on="raceId")[dup_mask.values].is_indy500.mean()
print(f"share of duplicate-key rows (Section 5) that belong to Indy 500 races: {dup_share:.1%}")

**Reading the evidence.** The Indy rows form a block of one-off, almost exclusively American drivers with larger fields; they make up a large share of the naive "American at a US circuit" home rows and of the early duplicate rows. Removing them makes the 1950s data *more* representative of Formula 1 racing, at the cost of a few hundred rows that were never informative about the 2014+ test era anyway.

### 6.2 DNQ / DNPQ / withdrawn / non-starters
In eras with more entrants than grid places (late 1980s–early 1990s especially), slow drivers *failed to (pre-)qualify* but still have a `results` row (`positionText = F`, status "Did not qualify"). Withdrawn entries (`W`) and 107%-rule exclusions behave the same way.

**Decision: separate them from the modeling table** (kept as `non_starters` for audit).

**Why this choice?**
* **Unit of analysis = "driver entering one race" and the prediction is made at the grid.** A non-qualifier is not on the grid; whether they are in the race is already known at the cutoff, so there is no prediction to make (the label is trivially 0).
* **They distort the base rate.** They inflate the negative class precisely in the eras with big fields, and make the "podium rate vs. grid" plot meaningless (`grid = 0` ≠ pole).
* **No positives are lost.** `positionOrder` ranks non-starters last, so a non-starter can never be on the podium — removing them changes the denominator, not the numerator.
* **Leakage-safe definition.** The flag uses only facts settled before the race (failed to qualify / withdrew / 107% rule) — **not** `laps` or the finishing status of drivers who raced, because dropping rows based on how the race went would itself select on the outcome. Drivers who were on the grid but retired on lap 0 therefore stay in (negative label).

In [ ]:
RES_DEDUP["is_non_starter"] = flag_non_starters(RES_DEDUP)
ns = RES_DEDUP[RES_DEDUP.is_non_starter & ~RES_DEDUP.is_indy500]
print(f"non-starter rows (outside Indy 500): {len(ns)} of {int((~RES_DEDUP.is_indy500).sum())}")
display(ns.groupby((ns.year // 10 * 10).rename("decade")).size().rename("non_starter_rows").to_frame().T)
print("positionText:", ns.positionText.value_counts().head(5).to_dict())
print("status:", ns.status.value_counts().head(6).to_dict())
print("podium rows among non-starters (must be 0):", int((ns.positionOrder <= 3).sum()))
if (ns.positionOrder <= 3).any():
    print("WARNING: a flagged non-starter has podium positionOrder - inspect these rows before trusting the rule:"); display(ns[ns.positionOrder <= 3].head())

# diagnostic only (NOT used to drop rows): grid==0 starters that completed 0 laps and are NOT flagged
amb = RES_DEDUP[(RES_DEDUP.grid == 0) & (RES_DEDUP.laps.fillna(0) == 0) & ~RES_DEDUP.is_non_starter & ~RES_DEDUP.is_indy500]
print("ambiguous grid=0 & 0 laps rows kept as starters (diagnostic):", len(amb))

RES_FINAL = RES_DEDUP[~RES_DEDUP.is_indy500 & ~RES_DEDUP.is_non_starter].reset_index(drop=True)
assert not RES_FINAL.duplicated(KEYS).any()

dd = RES_DEDUP[~RES_DEDUP.is_indy500].copy(); dd["decade"] = dd.year // 10 * 10
rate = pd.DataFrame({"incl. non-starters": dd.groupby("decade").apply(lambda d: d.podium.mean(), include_groups=False),
                     "starters only": dd[~dd.is_non_starter].groupby("decade").apply(lambda d: d.podium.mean(), include_groups=False)})
ax = rate.plot(kind="bar", figsize=(10, 4)); ax.set_title("Podium base rate per decade: with vs without non-starters"); ax.set_ylabel("podium rate")
plt.show()
print("modeling population now:", len(RES_FINAL), "rows")

---
# 7. EDA **after** cleaning & before/after comparison

We recompute the Section-3 metrics on the cleaned modeling population (`RES_FINAL`) and put them next to the raw ones. The goal is to show that each cleaning step **fixes an anomaly without shifting the distributions it should not touch**.

### Why this choice? (what counts as "success")
| Step | Anomaly | Evidence that it worked | Evidence of *no bias* |
|---|---|---|---|
| `\N` → `NaN` | numeric/date columns stored as text | dtypes now numeric/datetime; 0 sentinels left | count of `\N` before == count of `NaN` after, column by column (nothing lost, nothing invented) |
| lap time → seconds | `"1:27.452"` not comparable | float seconds, plausible range | unparsable values are counted and listed (expected ~0; any remainder is inspected) |
| de-duplication | rows per race > unique drivers; some races with > 3 "podium" rows | rows = unique drivers; exactly 3 podium rows per race (except documented exceptions) | modern eras unchanged; only early decades move; podium positives lost are counted |
| Indy / non-starters | wrong population | field size & podium rate become coherent with the 3/N benchmark | no podium rows lost to the non-starter rule |

In [ ]:
SEASON_CLEAN = season_stats(RES_FINAL)
cmp_year = SEASON_RAW[["mean_field", "podium_rate"]].join(SEASON_CLEAN[["mean_field", "podium_rate"]], lsuffix="_raw", rsuffix="_clean")

# sentinel preservation check: literal \N before  vs  NaN after (column by column)
chk = []
for t, df in RAW.items():
    for c in df.columns:
        n_s = int(((df[c] == SENT) | (df[c].str.strip() == "")).sum())    # literal \N or blank cell
        if n_s == 0: continue
        dst = LAPTIME_COLS.get(t, {}).get(c, c)
        if dst in CLEAN[t].columns:
            n_na = int(CLEAN[t][dst].isna().sum())
            chk.append(dict(table=t, column=c, mapped_to=dst, sentinels_before=n_s, nan_after=n_na, delta=n_na - n_s))
CHK = pd.DataFrame(chk)
bad = CHK[CHK.delta != 0]
print(f"columns with missing markers: {len(CHK)} | columns where NaN-after != markers-before: {len(bad)}")
if len(bad): display(bad)
# DIAGNOSTIC: values that were NOT a missing marker but are still NaN after conversion (unparsable text)
for _, r in bad.iterrows():
    src = r["column"]; dst = r["mapped_to"]
    m = ~((RAW[r["table"]][src] == SENT) | (RAW[r["table"]][src].str.strip() == "")) & CLEAN[r["table"]][dst].isna()
    print(f"{r['table']}.{src}: {int(m.sum())} unparsable non-missing values, e.g.", RAW[r["table"]].loc[m, src].value_counts().head(8).to_dict())

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes[0, 0].plot(cmp_year.index, cmp_year.mean_field_raw, label="raw (rows / race)"); axes[0, 0].plot(cmp_year.index, cmp_year.mean_field_clean, label="clean (starters / race)")
axes[0, 0].set_title("Mean field size per season"); axes[0, 0].legend()
axes[0, 1].plot(cmp_year.index, cmp_year.podium_rate_raw, label="raw"); axes[0, 1].plot(cmp_year.index, cmp_year.podium_rate_clean, label="clean")
axes[0, 1].plot(SEASON_CLEAN.index, SEASON_CLEAN.benchmark_3_over_N, "k:", label="3 / clean field size"); axes[0, 1].set_title("Podium rate per season"); axes[0, 1].legend()

pr_raw = RAW_RES.groupby("raceId").positionOrder.apply(lambda s: int((s <= 3).sum())).value_counts().rename("raw")
pr_cln = RES_FINAL.groupby("raceId").podium.sum().astype(int).value_counts().rename("clean")
pr = pd.concat([pr_raw, pr_cln], axis=1).fillna(0).sort_index()
pr.plot(kind="bar", ax=axes[1, 0]); axes[1, 0].set_title("# podium-labelled rows per race (3, or more when a podium car was shared)"); axes[1, 0].set_xlabel("podium rows in race"); axes[1, 0].set_ylabel("# races")

top = CHK.sort_values("sentinels_before", ascending=False).head(12).assign(label=lambda d: d.table + "." + d.column)
x = np.arange(len(top)); axes[1, 1].bar(x - .2, top.sentinels_before, .4, label="literal \\N before"); axes[1, 1].bar(x + .2, top.nan_after, .4, label="NaN after")
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(top.label, rotation=75, ha="right"); axes[1, 1].set_yscale("log"); axes[1, 1].set_title("Missing markers before vs NaN after"); axes[1, 1].legend()
plt.show()
print("races whose podium-row count != 3 after cleaning:", int((pr_cln.drop(3, errors='ignore')).sum()), "of", int(pr_cln.sum()))

In [ ]:
# typing impact: distributions that only became possible after conversion to seconds
R = CLEAN["results"].merge(CLEAN["races"][["raceId", "year"]], on="raceId"); Qs = CLEAN["qualifying"].merge(CLEAN["races"][["raceId", "year"]], on="raceId")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(R.fastestLapTime_s.dropna().clip(30, 200), bins=60, ax=axes[0]); axes[0].set_title("fastestLapTime_s (seconds)")
sns.histplot(Qs.q3_s.dropna().clip(30, 200), bins=60, ax=axes[1], color="tab:green"); axes[1].set_title("q3_s (seconds)")
Qs.assign(dec=Qs.year // 10 * 10).dropna(subset=["q1_s"]).boxplot(column="q1_s", by="dec", ax=axes[2]); axes[2].set_title("q1_s by decade (circuit mix varies)"); plt.suptitle("")
plt.show()
for c, d in [("fastestLapTime_s", R), ("q1_s", Qs), ("q2_s", Qs), ("q3_s", Qs)]:
    s = d[c].dropna(); print(f"{c:17s} n={len(s):6d} min={s.min():7.3f} median={s.median():7.3f} max={s.max():8.3f} | >300s: {int((s > 300).sum())}")

# the cost of cleaning: rows removed per reason and the podium positives they carried
n_pod = lambda d: int((d.positionOrder <= 3).sum())      # raw-row count, so the accounting tallies exactly
ind = RES_DEDUP[RES_DEDUP.is_indy500]; nsr = RES_DEDUP[~RES_DEDUP.is_indy500 & RES_DEDUP.is_non_starter]
acc = pd.DataFrame([
    ("typed results (start)", len(RES_ALL), n_pod(RES_ALL)),
    ("- duplicate rows dropped", len(DROPPED), n_pod(DROPPED)),
    ("- Indy 500 rows dropped", len(ind), n_pod(ind)),
    ("- non-starter rows dropped", len(nsr), n_pod(nsr)),
    ("= modeling population (kept-row positionOrder<=3)", len(RES_FINAL), n_pod(RES_FINAL)),
    ("   + label uplift: driver's best row on the podium (shared drives)", 0, int(RES_FINAL.podium.sum()) - n_pod(RES_FINAL))],
    columns=["step", "rows", "podium_rows"])
acc["share_of_start_rows"] = (acc.rows / len(RES_ALL)).round(4)
display(acc)

dec_raw = RAW_RES.assign(dec=RAW_RES.year // 10 * 10).groupby("dec").apply(lambda d: pd.Series({"rows_raw": len(d), "podium_rate_raw": (d.positionOrder <= 3).mean()}), include_groups=False)
dec_cln = RES_FINAL.assign(dec=RES_FINAL.year // 10 * 10).groupby("dec").apply(lambda d: pd.Series({"rows_clean": len(d), "podium_rate_clean": d.podium.mean()}), include_groups=False)
display(dec_raw.join(dec_cln).assign(rate_shift=lambda d: d.podium_rate_clean - d.podium_rate_raw).round(4))

### Interpretation: cleaning fixed anomalies without biasing distributions
* **Sentinels:** the check compares missing markers (`\N` or blank) before with NaN after, column by column. A `delta` of 0 means no cell was dropped, filled or invented. A positive `delta` means some non-missing text could not be parsed (e.g. an odd lap-time format); the diagnostic above prints those exact values so they can be fixed or consciously accepted — they become NaN, never an invented number.
* **Duplicates:** after de-duplication the number of rows per race equals the number of distinct drivers, and almost every race has **exactly three** podium-labelled rows (the few that do not are real sporting oddities such as shared/ties and are listed by the check above, not hidden). Before, races with shared drives could show more than three.
* **No distribution distortion where it should not matter:** the decade table shows the podium-rate shift is essentially confined to the early decades (duplicates + Indy + big-field non-starters); the modern decades — our validation/test region — are unchanged. Podium positives removed by the rules are listed in the accounting table (non-starters remove **zero**; duplicate/Indy removal removes only rows that are not the unique driver-race pair or not F1).
* **Lap times** now sit in plausible ranges (tens of seconds for a lap; no conversion artefacts), which makes `gap to pole (%)` and similar features computable by P3.

---
# 8. Hand-off artifact: `load_clean()` and the data dictionary

### What `load_clean()` does (and why encapsulation matters)
It wraps the whole pipeline — read raw → sentinels/typing → Indy rule → dedup rule → non-starter rule → qualifying join → **split into a pre-race `model_base` and a post-race `outcomes` table**.

**Why this choice?**
* **One definition of "the data".** If P2 (joins), P3 (features) and P4 (models) each clean their own way, a tiny difference (one keeps Indy, one drops DNQs) silently changes row counts and metrics, and results are no longer comparable. A single function guarantees that all four of us start from *byte-identical* tables, and any change is a one-place edit reviewed in a PR.
* **Leakage is prevented structurally.** Post-race columns (`position`, `points`, `laps`, `status`, fastest lap, …) live only in `outcomes`. `model_base` contains pre-race information plus the target `podium`; the function **asserts** that no forbidden column is present, that keys are unique and non-null, and that both tables have the same rows. Standings tables are post-race snapshots and are returned in `tables` with a documented warning: they must be shifted to "previous round" before use.
* **Nothing is fitted.** There is no imputation, scaling or encoding inside `load_clean()`, so there is nothing that could be fitted on test data — those steps belong to P3's train-only pipeline.
* **Deterministic.** Seeds set; stable sort order; no randomness in any step.

In [ ]:
# ---- Column contracts: what is allowed to become a FEATURE vs. what is outcome-only ---------------
PRE_RACE_COLS = ["raceId", "driverId", "constructorId", "year", "round", "date", "circuitId", "number",
                 "grid", "pitlane_start", "has_qualifying", "qual_position", "q1_s", "q2_s", "q3_s"]
OUTCOME_COLS = ["raceId", "driverId", "resultId", "positionOrder", "podium", "positionText", "position",
                "points", "laps", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime_s",
                "fastestLapSpeed", "statusId", "status", "n_raw_rows", "shared_drive", "best_positionOrder_raw"]
FORBIDDEN_IN_BASE = {"positionOrder", "positionText", "position", "points", "laps", "time", "milliseconds",
                     "fastestLap", "rank", "fastestLapTime", "fastestLapTime_s", "fastestLapSpeed",
                     "statusId", "status", "resultId", "n_raw_rows", "shared_drive", "best_positionOrder_raw"}


def prep_qualifying(q: pd.DataFrame):
    """Qualifying should be unique per (raceId, driverId); if not, keep the highest qualifyId."""
    q = q.sort_values("qualifyId")
    n_dup = int(q.duplicated(KEYS, keep="last").sum())
    q = q.drop_duplicates(KEYS, keep="last").rename(columns={"position": "qual_position"})
    return q[KEYS + ["qual_position", "q1_s", "q2_s", "q3_s"]], n_dup


def load_clean(data_dir=None, exclude_indy500: bool = True, drop_non_starters: bool = True,
               exclude_shared_drives: bool = False) -> dict:
    """THE handoff function. Raw CSV -> typed -> Indy rule -> dedup -> non-starter rule -> split into
    a PRE-RACE base table and a POST-RACE outcomes table (never use outcomes columns as features).
    exclude_shared_drives=True additionally drops every (race, driver) that had >1 raw result row (1950s
    car swaps); default False keeps them (flagged shared_drive=True in `outcomes`). The label `podium` uses the
    driver's best result across his rows; the pre-race columns come from ONE row chosen without any outcome.

    Returns dict with:
      model_base : one row per (raceId, driverId): pre-race columns + target `podium`
      outcomes   : same grain, post-race columns (status, points, positions...) for DE questions / audits
      non_starters, dup_dropped : rows removed, kept for audit
      tables     : every other typed table (races incl. is_indy500, circuits, drivers, constructors, status,
                   qualifying, *_standings (post-race! shift before use), lap_times, pit_stops, ...)
      meta       : row accounting for every filtering step
    """
    set_all_seeds(SEED)
    data_dir = Path(data_dir) if data_dir else resolve_data_dir()
    T, _ = clean_all(read_raw_tables(data_dir))
    T["races"] = flag_races(T["races"], T["circuits"])

    res = (T["results"]
           .merge(T["races"][["raceId", "year", "round", "date", "circuitId", "is_indy500"]],
                  on="raceId", how="inner", validate="m:1")
           .merge(T["status"], on="statusId", how="left", validate="m:1"))
    meta = {"raw_result_rows": len(T["results"]), "rows_after_race_join": len(res)}

    if exclude_indy500:
        meta["indy500_rows_removed"] = int(res["is_indy500"].sum())
        res = res[~res["is_indy500"]]
    check_dedup_is_outcome_free(res)
    res, dup_dropped = dedup_results(res)
    meta["duplicate_rows_removed"] = len(dup_dropped)
    res["is_non_starter"] = flag_non_starters(res)
    non_starters = res[res["is_non_starter"]].copy()
    meta["non_starter_rows_removed"] = len(non_starters) if drop_non_starters else 0
    if drop_non_starters:
        res = res[~res["is_non_starter"]]
    res = res.reset_index(drop=True)
    meta["shared_drive_rows_removed"] = 0
    if exclude_shared_drives:      # optional: the kept row of a shared drive may carry the teammate's car
        meta["shared_drive_rows_removed"] = int(res["shared_drive"].sum())
        res = res[~res["shared_drive"]].reset_index(drop=True)
    meta["final_rows"] = len(res)

    q, meta["qualifying_duplicate_rows_dropped"] = prep_qualifying(T["qualifying"])
    res["pitlane_start"] = res["grid"].eq(0).fillna(False).astype(bool)   # grid==0 among actual starters
    base = res.merge(q, on=KEYS, how="left", validate="1:1", indicator="_q")
    base["has_qualifying"] = base["_q"].eq("both")
    base = base.drop(columns="_q")
    outcomes = base[OUTCOME_COLS].copy()
    model_base = base[PRE_RACE_COLS].copy()
    model_base["podium"] = outcomes["podium"].values

    # ---- hard guarantees every teammate can rely on -----------------------------------------------
    assert not model_base.duplicated(KEYS).any(), "grain violated: duplicate (raceId, driverId)"
    assert model_base[KEYS].notna().all().all(), "null key in modeling table"
    assert not (FORBIDDEN_IN_BASE & set(model_base.columns)), "post-race column leaked into base table"
    assert len(model_base) == len(outcomes) == meta["final_rows"]
    return dict(model_base=model_base, outcomes=outcomes, non_starters=non_starters,
                dup_dropped=dup_dropped, tables=T, meta=meta)

In [ ]:
CL = load_clean(DATA_DIR)
MB, OUT = CL["model_base"], CL["outcomes"]
print("meta:", CL["meta"])
display(MB.head(5)); display(MB.dtypes.to_frame("dtype").T)

# the function must reproduce the step-by-step result above exactly
assert set(map(tuple, MB[KEYS].to_numpy())) == set(map(tuple, RES_FINAL[KEYS].to_numpy()))
assert len(MB) == len(RES_FINAL) and MB.podium.sum() == int(RES_FINAL.podium.sum())
assert not MB.duplicated(KEYS).any()
print(f"\nload_clean OK: {len(MB)} rows, {MB.raceId.nunique()} races, {MB.driverId.nunique()} drivers, "
      f"{MB.year.min()}-{MB.year.max()}, podium rate {MB.podium.mean():.3%}")
# quick sanity on the agreed split (P3 owns the real split)
split = np.select([MB.year <= 2019, MB.year.between(2020, 2021)], ["train (<=2019)", "val (2020-21)"], "test (>=2022)")
display(MB.groupby(split).podium.agg(rows="size", podium_rate="mean").round(4))

In [ ]:
# ---- Data dictionary ------------------------------------------------------------------------------
DESC = {
 "raceId": ("key", "Race id (FK -> races). Part of the grain."),
 "driverId": ("key", "Driver id (FK -> drivers). Part of the grain."),
 "constructorId": ("pre-race", "Constructor of the KEPT row (chosen with pre-race info only; for shared drives possibly the teammate's car). Renames create new ids (history resets)."),
 "year": ("pre-race", "Season (from races)."), "round": ("pre-race", "Round number within the season."),
 "date": ("pre-race", "Race date (datetime64)."), "circuitId": ("pre-race", "Circuit id (FK -> circuits)."),
 "number": ("pre-race", "Car number (Int64; NaN when unrecorded)."),
 "grid": ("pre-race", "Starting grid slot AFTER penalties. 0 = no grid slot (pit-lane start among starters); raw value kept."),
 "pitlane_start": ("pre-race", "True if grid == 0 among rows that actually started (non-starters were removed first)."),
 "has_qualifying": ("pre-race", "True if a qualifying row exists for (race, driver). Coverage drifts by era -> see Section 3."),
 "qual_position": ("pre-race", "Qualifying classification (before grid penalties). NaN when no qualifying data."),
 "q1_s": ("pre-race", "Q1 best lap, seconds (float). NaN if no session / not set."),
 "q2_s": ("pre-race", "Q2 best lap, seconds. NaN structurally if eliminated in Q1 or pre-2006."),
 "q3_s": ("pre-race", "Q3 best lap, seconds. NaN structurally if eliminated earlier or pre-2006."),
 "podium": ("TARGET", "1 if the driver's best positionOrder over all his rows in the race is <= 3 (equals positionOrder<=3 except shared drives). int8."),
 "resultId": ("outcome", "Source result row id of the kept row."),
 "positionOrder": ("outcome", "Final ranking of every entrant, always filled. Source of the target."),
 "positionText": ("outcome", "Official position text / letter code (R, D, E, W, F, N)."),
 "position": ("outcome", "Classified position; NaN if not classified."),
 "points": ("outcome", "Championship points scored in this race."), "laps": ("outcome", "Laps completed."),
 "time": ("outcome", "Race time text as published."), "milliseconds": ("outcome", "Race time in ms."),
 "fastestLap": ("outcome", "Lap number of driver's fastest lap."), "rank": ("outcome", "Fastest-lap rank in the race."),
 "fastestLapTime_s": ("outcome", "Fastest lap time in seconds (converted from m:ss.fff)."),
 "fastestLapSpeed": ("outcome", "Fastest lap average speed (km/h)."),
 "statusId": ("outcome", "FK -> status (why the race ended)."), "status": ("outcome", "Status text: Finished, +1 Lap, Engine, Collision, ..."),
 "n_raw_rows": ("audit", "How many raw result rows this (race, driver) had before de-duplication."),
 "shared_drive": ("audit", "True if n_raw_rows > 1 (1950s car swap). Pre-race columns come from one row chosen by pre-race convention (lowest grid), so grid/constructorId may be the teammate's car; drop with load_clean(exclude_shared_drives=True) if unwanted."),
 "best_positionOrder_raw": ("audit", "Best positionOrder among the driver's raw rows; the source of the label podium."),
}
RULES = {"grid": "\\N->NaN, Int64; non-starters removed", "q1_s": "m:ss.fff -> seconds; \\N->NaN", "q2_s": "m:ss.fff -> seconds; \\N->NaN",
         "q3_s": "m:ss.fff -> seconds; \\N->NaN", "date": "string -> datetime64", "fastestLapTime_s": "m:ss.fff -> seconds; \\N->NaN",
         "constructorId": "dedup keep-rule: pre-race only (grid>0, lowest grid, constructorId, number); outcome never used",
         "podium": "best positionOrder over the driver's rows <= 3 (outcome used for the LABEL only)", "qual_position": "qualifying.position; dedup on (race,driver) keep highest qualifyId"}

def data_dictionary(clean: dict) -> pd.DataFrame:
    rows = []
    for tname in ("model_base", "outcomes"):
        df = clean[tname]
        for c in df.columns:
            role, desc = DESC.get(c, ("?", ""))
            rows.append(dict(table=tname, column=c, dtype=str(df[c].dtype), role=role, description=desc,
                             cleaning_rule=RULES.get(c, "\\N->NaN; typed" if role != "key" else "typed Int64"),
                             pct_missing=round(100 * df[c].isna().mean(), 2)))
    return pd.DataFrame(rows)

DD = data_dictionary(CL)
assert not (DD.role == "?").any(), DD[DD.role == "?"]
display(DD)
DD.to_csv(OUTPUT_DIR / "data_dictionary.csv", index=False)

def export_clean(clean: dict, out_dir) -> None:
    """v0 hand-off for teammates who prefer files (git-ignored). Prefer load_clean() over these CSVs."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    clean["model_base"].to_csv(out_dir / "model_base.csv", index=False)
    clean["outcomes"].to_csv(out_dir / "outcomes.csv", index=False)
export_clean(CL, OUTPUT_DIR / "clean_v0")
print("wrote", OUTPUT_DIR / "data_dictionary.csv", "and", OUTPUT_DIR / "clean_v0/")

### Rules for teammates (also in the data dictionary)
* **Features may only come from `model_base`** (+ other pre-race tables joined by *earlier* rounds). Columns with role `outcome`/`audit` are never features.
* `*_standings` in `CL["tables"]` are **after-the-race** snapshots: use the previous round (`shift`) — first race of a season has no standings.
* Rolling form must use `groupby(...).shift(1)`.
* Shared drives (`shared_drive == True`, ≈0.3% of rows, all ≤ 1957): kept by default; use `load_clean(exclude_shared_drives=True)` to drop them. P3 decides and documents.
* Fit imputers / scalers / encoders on the **train** years only (≤ 2019).

---
# 9. Notebook integration & Kaggle "Run All" verification

### Merging everyone's sections into `main.ipynb`
Each person keeps a draft notebook (`notebooks/p1_data.ipynb`, `p2_de.ipynb`, `p3_features.ipynb`, `p4_models.ipynb`, plus `p1_inference.ipynb`). The script `tools/merge_notebooks.py` concatenates them **in the fixed order below**, keeps only the *first* notebook's setup cell (seeds + `DATA_DIR`), strips outputs, and fails if a section re-defines `DATA_DIR`/seeds or contains `!pip` / absolute local paths:

```bash
python tools/merge_notebooks.py notebooks/p1_data.ipynb notebooks/p2_de.ipynb \
       notebooks/p3_features.ipynb notebooks/p4_models.ipynb notebooks/p1_inference.ipynb -o main.ipynb
bash tools/run_all_check.sh main.ipynb      # headless Run All; non-zero exit code on any error
```
**Order of sections:** setup → P1 audit/clean → P2 data-engineering questions → P3 features/split/preprocessing → P4 models → P3 ablation → P4 XAI → P1 inference → final self-check cell (next).
**On Kaggle:** *Add Data* → attach `rohanrao/formula-1-world-championship-1950-2020`; upload `main.ipynb`; if `shap`/`lime` are missing, one guarded cell near the top may run `%pip install -q shap lime` (requires Internet **on** for that session); then *Run All*, check *Save Version → Save & Run All*, and make the notebook link accessible.

### Why this choice? (strict top-to-bottom order and seed locking)
* **Kaggle runs the notebook fresh, top to bottom, in a new kernel.** Anything that worked only because a cell was run out of order, or because a variable survived from an earlier interactive session (a "hidden state" bug), fails at evaluation. Strict order means every cell depends only on cells above it — the property `run_all_check.sh` verifies.
* **Seeds make results reproducible and comparable.** Random forests, gradient boosting, NN weight init / dropout, SHAP background sampling and LIME perturbations are all stochastic. A graders' re-run must reproduce the numbers in our report and the ablation deltas must be real effects, not seed noise. We fix a single `SEED` in the first code cell (`set_all_seeds`) and pass `random_state=SEED` explicitly to every estimator.
* **Path resolution, not paths.** `resolve_data_dir()` finds the CSVs on Kaggle or locally, so the same file passes on three laptops and on Kaggle.

## Part 2 — Data-engineering questions Q1–Q3 *(owner: P2 — insert here)*
## Part 3 — Features, split, preprocessing experiments *(owner: P3 — insert here)*
## Part 4 — Models & metrics *(owner: P4 — insert here)*
## Part 5 — Feature-group ablation *(P3)* and XAI *(P4)*
## Part 6 — Inference function `predict_podium(record)` *(owner: P1, after P4's model is frozen)*

In [ ]:
# ---- Final Run-All self-check (keep this as the LAST cell of main.ipynb) --------------------------
def assert_run_all_ready():
    problems = []
    if SEED != 42: problems.append("SEED changed")
    if not Path(DATA_DIR).exists(): problems.append("DATA_DIR missing")
    for name in ("RAW", "CLEAN", "CL"):
        if name not in globals(): problems.append(f"{name} not defined (cells executed out of order?)")
    if not CL["model_base"].duplicated(KEYS).sum() == 0: problems.append("grain violated")
    if FORBIDDEN_IN_BASE & set(CL["model_base"].columns): problems.append("leakage columns in base table")
    again = load_clean(DATA_DIR)["model_base"]                      # determinism: same input -> identical output
    if not again.reset_index(drop=True).equals(CL["model_base"].reset_index(drop=True)): problems.append("load_clean not deterministic")
    assert not problems, problems
    print(f"RUN-ALL CHECK PASSED in {(_time.time() - _T0) / 60:.1f} min | rows={len(CL['model_base'])} | seed={SEED} | DATA_DIR={DATA_DIR}")

assert_run_all_ready()